# Introduction

Electronic health records (EHR) provide an important source of structured clinical information that can be used to study patient conditions, treatment patterns, and health outcomes. Machine learning techniques are being used more and more to evaluate large-scale patient datasets in order to find significant patterns, assist clinical decision-making, and enhance medical risk prediction as digital healthcare systems expand quickly. In this project, we examine a synthetic healthcare dataset called the *Patient Records Dataset*, which was first made available on Kaggle and contains records for 100000 patients across 15 medical conditions. This dataset incorporates several tables that describe demographics, diagnoses, laboratory measures, prescription drugs, and clinical outcomes while simulating actual hospital information systems.

The dataset used in this project consists of five relational tables:

patients.csv: demographic information such as age, gender, and patient identifiers
diagnoses.csv: diagnosed medical conditions associated with each patient
medications.csv: prescribed medications and treatment records
lab_results.csv: laboratory test measurements reflecting physiological status
outcomes.csv: clinical outcome indicators such as hospitalization status and disease progression

When combined, these tables offer a thorough depiction of patient health trajectories and enable us to investigate connections between patient attributes, therapies, test results, and results.

This project's goal is to use Python data science tools like *NumPy*, *Pandas*, *Matplotlib*, *Seaborn*, and *Scikit-learn* to conduct a comprehensive machine learning analysis on this multi-table healthcare dataset. Following feature engineering, integration, and data cleansing, we look into activities related to structure discovery and predictive modeling.

In particular, this project tackles two related machine learning issues:

1. *Supervised learning task:*:
   Using laboratory measures, diagnoses, medication history, and demographic information, we build predictive models to forecast clinical outcomes for patients. To assess predicted performance, several supervised learning techniques are used and contrasted.

2. *Unsupervised learning task:*:
   To uncover hidden structures in the patient population and find possible patient subgroups with comparable clinical traits, we use dimensionality reduction and clustering approaches.

Through these analyses, the project shows how machine learning methods may be used to support patient classification and outcome prediction in structured healthcare data. The findings demonstrate the value of integrating laboratory, therapy, diagnostic, and demographic data when developing predictive models in healthcare analytics.

# Data Loading and Basic Cleaning

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold

# 1. Load all tables
patients = pd.read_csv("patients.csv")
diagnoses = pd.read_csv("diagnoses.csv")
medications = pd.read_csv("medications.csv")
lab_results = pd.read_csv("lab_results.csv")
outcomes = pd.read_csv("outcomes.csv")

# 2. Standardize patient_id format
for df in [patients, diagnoses, medications, lab_results, outcomes]:
    df["patient_id"] = df["patient_id"].astype(str)

# 3. Check duplicates
for name, df in zip(["patients", "diagnoses", "medications", "lab_results", "outcomes"],
                    [patients, diagnoses, medications, lab_results, outcomes]):
    print(f"{name} duplicate rows: {df.duplicated().sum()}")
    df.drop_duplicates(inplace=True)

# 4. Check missing values
print("Missing value ratio per table:")
for name, df in zip(["patients", "diagnoses", "medications", "lab_results", "outcomes"],
                    [patients, diagnoses, medications, lab_results, outcomes]):
    print(f"\n{name}:")
    print(df.isnull().mean().sort_values(ascending=False).head(10))

patients duplicate rows: 0
diagnoses duplicate rows: 0
medications duplicate rows: 0
lab_results duplicate rows: 0
outcomes duplicate rows: 0
Missing value ratio per table:

patients:
patient_id                   0.0
age                          0.0
dx_osteoarthritis            0.0
dx_hypothyroidism            0.0
dx_anxiety                   0.0
dx_depression                0.0
dx_asthma                    0.0
dx_copd                      0.0
dx_chronic_kidney_disease    0.0
dx_atrial_fibrillation       0.0
dtype: float64

diagnoses:
secondary_diagnoses    0.41566
secondary_icd10s       0.41566
patient_id             0.00000
visit_date             0.00000
visit_type             0.00000
primary_diagnosis      0.00000
primary_icd10          0.00000
provider_specialty     0.00000
dtype: float64

medications:
patient_id       0.0
medication       0.0
dose             0.0
unit             0.0
frequency        0.0
indication       0.0
start_date       0.0
duration_days    0.0
is_generic    

In [2]:
# 5. Handle critical missing values
# ----------------------
# diagnoses: 41.5% missing in secondary_diagnoses & secondary_icd10s
# ----------------------
# Fill with empty string (meaning "no secondary diagnoses")
diagnoses["secondary_diagnoses"] = diagnoses["secondary_diagnoses"].fillna("")
diagnoses["secondary_icd10s"] = diagnoses["secondary_icd10s"].fillna("")

# ----------------------
# outcomes: 85% missing in days_to_readmission
# ----------------------
# 85% missing = this column is almost useless → drop it
outcomes = outcomes.drop(columns=["days_to_readmission"])

# ----------------------
# patients & medications & lab_results: NO MISSING VALUES → DO NOTHING
# ----------------------

# Sub-table Feature Engineering

## About <patients.csv>

In [3]:
patients

,patient_id,age,sex,bmi,systolic_bp,diastolic_bp,heart_rate,temperature_f,smoking_status,alcohol_use,...,dx_heart_failure,dx_atrial_fibrillation,dx_chronic_kidney_disease,dx_copd,dx_asthma,dx_depression,dx_anxiety,dx_hypothyroidism,dx_osteoarthritis,dx_type1_diabetes
0,P0000001,66,M,23.5,148,81,64,98.4,former,light,...,0,0,1,0,0,0,0,0,1,0
1,P0000002,75,M,24.8,158,86,45,99.5,never,moderate,...,0,0,0,0,1,0,0,0,1,0
2,P0000003,82,M,17.8,135,57,90,98.2,never,light,...,0,0,1,0,0,0,0,0,0,0
3,P0000004,73,F,28.1,118,83,102,98.9,former,moderate,...,0,0,0,0,0,0,0,0,0,0
4,P0000005,86,F,30.6,156,81,56,98.7,never,heavy,...,0,0,0,0,1,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P0099996,37,M,24.9,106,101,93,98.9,never,light,...,0,0,0,0,0,0,0,0,0,0
99996,P0099997,37,F,27.8,80,95,63,98.7,never,moderate,...,0,0,0,0,0,0,0,1,0,0
99997,P0099998,50,F,28.5,115,83,53,99.2,never,moderate,...,0,0,0,0,0,0,0,0,0,0
99998,P0099999,85,M,34.6,149,73,65,98.5,never,none,...,0,0,0,1,1,0,0,0,1,0


In [4]:
# 1. One-hot encoding for categorical variables
original_cols = patients.columns.tolist() # List of original column names
cat_cols_pat = patients.select_dtypes(exclude=[np.number]).columns.drop("patient_id") # List of non-numeric column names
for col in cat_cols_pat:
    if patients[col].nunique() > 10: # 10 is empirical value
        print(f"High cardinality column {col}, unique values: {patients[col].nunique()}")
        patients.drop(col, axis=1, inplace=True)
    else:
        patients = pd.get_dummies(patients, columns=[col], prefix=col, drop_first=True)

# 2. Standardize numerical features
scaler = StandardScaler()
num_cols_pat = patients.select_dtypes(include=[np.number]).columns.drop(["patient_id"], errors="ignore") # List of numeric column names
patients[num_cols_pat] = scaler.fit_transform(patients[num_cols_pat])

new_cols = patients.columns.tolist() # List of new column names
added_cols = [col for col in new_cols if col not in original_cols]
patients[added_cols]

,sex_M,smoking_status_former,smoking_status_never,alcohol_use_light,alcohol_use_moderate,alcohol_use_none,exercise_level_moderate,exercise_level_sedentary,exercise_level_vigorous,insurance_type_medicaid,insurance_type_medicare,insurance_type_tricare,insurance_type_uninsured
0,True,True,False,True,False,False,True,False,False,False,True,False,False
1,True,False,True,False,True,False,True,False,False,False,False,False,False
2,True,False,True,True,False,False,True,False,False,False,True,False,False
3,False,True,False,False,True,False,True,False,False,True,False,False,False
4,False,False,True,False,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,True,False,True,True,False,False,False,False,True,True,False,False,False
99996,False,False,True,False,True,False,False,False,True,False,False,False,False
99997,False,False,True,False,True,False,False,True,False,False,False,False,False
99998,True,False,True,False,False,True,False,False,False,False,False,False,False


## About <diagnoses.csv>

In [5]:
diagnoses

,patient_id,visit_date,visit_type,primary_diagnosis,primary_icd10,secondary_diagnoses,secondary_icd10s,provider_specialty
0,P0000001,2018-07-26,telehealth,hyperlipidemia,E78.5,chronic_kidney_disease,N18.3,internal_medicine
1,P0000001,2020-04-09,outpatient,hyperlipidemia,E78.5,chronic_kidney_disease|osteoarthritis,N18.3|M19.90,nephrology
2,P0000001,2023-08-12,emergency,chronic_kidney_disease,N18.3,hyperlipidemia,E78.5,internal_medicine
3,P0000001,2022-11-04,outpatient,osteoarthritis,M19.90,hyperlipidemia|chronic_kidney_disease,E78.5|N18.3,internal_medicine
4,P0000002,2020-07-29,outpatient,osteoarthritis,M19.90,type2_diabetes,E11.9,nephrology
...,...,...,...,...,...,...,...,...
274587,P0099999,2018-04-24,outpatient,copd,J44.1,osteoarthritis,M19.90,nephrology
274588,P0100000,2020-07-01,inpatient,osteoarthritis,M19.90,,,internal_medicine
274589,P0100000,2019-08-30,inpatient,asthma,J45.909,hypertension,I10,internal_medicine
274590,P0100000,2022-08-28,emergency,asthma,J45.909,hypertension|osteoarthritis,I10|M19.90,cardiology


In [6]:
# 1. Split secondary ICD codes and diagnoses into list
diagnoses["secondary_icd_list"] = diagnoses["secondary_icd10s"].str.split("|")
diagnoses["secondary_diag_list"] = diagnoses["secondary_diagnoses"].str.split("|")

# 2. Extract the four core columns
primary_mapping = diagnoses[["primary_icd10", "primary_diagnosis"]].drop_duplicates()
primary_mapping.columns = ["icd10_code", "diagnoses_name"]
diagnoses_exploded = diagnoses.explode(["secondary_icd_list", "secondary_diag_list"])
secondary_mapping = diagnoses_exploded[["secondary_icd_list", "secondary_diag_list"]].rename(
    columns={
        "secondary_icd_list": "icd10_code",
        "secondary_diag_list": "diagnoses_name"
    }
).drop_duplicates()

# 3. Merge the diagnostic mappings and generate a complete comparison table
full_mapping = pd.concat([primary_mapping, secondary_mapping], ignore_index=True)
full_mapping = full_mapping.drop_duplicates().reset_index(drop=True)
full_mapping = full_mapping[
    (full_mapping["icd10_code"].str.strip() != "") &
    (full_mapping["diagnoses_name"].str.strip() != "")
] # Delete blank lines

print("ICD Coding - Diagnosis Name Mapping Table：")
print(full_mapping)
full_mapping.to_csv("icd_code_to_name_mapping.csv", index=False, encoding="utf-8")

ICD Coding - Diagnosis Name Mapping Table：
   icd10_code           diagnoses_name
0       E78.5           hyperlipidemia
1       N18.3   chronic_kidney_disease
2      M19.90           osteoarthritis
3       E11.9           type2_diabetes
4         I10             hypertension
5      I25.10  coronary_artery_disease
6       E66.9                  obesity
7       F32.9               depression
8       J44.1                     copd
9      I48.91      atrial_fibrillation
10      F41.9                  anxiety
11    J45.909                   asthma
12      E03.9           hypothyroidism
13      E10.9           type1_diabetes
14      I50.9            heart_failure


In [7]:
# 1. Aggregate visit features
diag_agg = diagnoses.groupby("patient_id").agg(
    total_visits=("visit_date", "count"),
    telehealth_visits=("visit_type", lambda x: sum(x == "telehealth")),
    emergency_visits=("visit_type", lambda x: sum(x == "emergency")),
    inpatient_visits=("visit_type", lambda x: sum(x == "inpatient")),
    outpatient_visits=("visit_type", lambda x: sum(x == "outpatient"))
).reset_index()

# 2. Collect all ICD codes (primary + secondary)
# 2.1 Extract the primary diagnosis ICD
primary_icds = diagnoses[["patient_id", "primary_icd10"]].rename(columns={"primary_icd10": "icd_code"})

# 2.2 Extract the secondary diagnosis ICD
secondary_icds = diagnoses[["patient_id", "secondary_icd_list"]].explode("secondary_icd_list")
secondary_icds = secondary_icds.rename(columns={"secondary_icd_list": "icd_code"})

# 2.3 Filter out null values
all_icds_long = pd.concat([primary_icds, secondary_icds], ignore_index=True)
all_icds_long = all_icds_long[all_icds_long["icd_code"] != ""].dropna(subset=["icd_code"])

# 3. Get top 15 frequent ICD codes
icd_series = all_icds_long["icd_code"].value_counts()
top_icds = icd_series.index

# 4. Create one-hot encoding for ICDs per patient
# 4.1 Group patients by patient ID and ICD number, and count the occurrence frequency of each ICD for each patient.
patient_icd_counts = all_icds_long.groupby(["patient_id", "icd_code"]).size().unstack(fill_value=0)

# 4.2 Fill in the missing ICD codes with 0.
patient_icd = patient_icd_counts.reindex(columns=top_icds, fill_value=0)

# 4.3 Reset the index and generate a standard DataFrame
icd_df = patient_icd.reset_index()

# 5. Convert to dataframe and merge with diag_agg
diag_agg = diag_agg.merge(icd_df, on="patient_id", how="left").fillna(0)

In [8]:
diag_agg

,patient_id,total_visits,telehealth_visits,emergency_visits,inpatient_visits,outpatient_visits,I10,E66.9,E78.5,N18.3,...,M19.90,F41.9,F32.9,I25.10,J45.909,J44.1,E03.9,I50.9,I48.91,E10.9
0,P0000001,4,1,1,0,2,0,0,4,4,...,2,0,0,0,0,0,0,0,0,0
1,P0000002,6,1,0,0,5,4,3,0,0,...,3,0,0,0,3,0,0,0,0,0
2,P0000003,1,0,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,P0000004,2,0,0,0,2,1,1,2,0,...,0,0,0,0,0,0,0,0,0,0
4,P0000005,1,0,0,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,P0099996,1,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
99996,P0099997,4,0,1,0,3,0,4,0,0,...,0,0,0,0,0,0,3,0,0,0
99997,P0099998,3,0,0,0,3,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
99998,P0099999,3,0,1,0,2,0,0,0,0,...,3,0,0,0,1,1,0,0,0,0


In [9]:
# 1. Sporadicity analysis
zero_ratio = (diag_agg[icd_cols] == 0).sum().sum() / (diag_agg[icd_cols].shape[0] * diag_agg[icd_cols].shape[1])
print(f"\n=== Analysis of ICD Feature Sparsity ===")
print(f"Proportion of zero values: {zero_ratio:.2%}")

# 2. Visualization of the distribution of the number of patients with non-zero ICD values
icd_cols = [col for col in diag_agg.columns if col not in ["patient_id"] + visit_cols]
icd_nonzero = diag_agg[icd_cols].astype(bool).sum(axis=0).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(icd_nonzero.index, icd_nonzero.values, color="skyblue", edgecolor="black", alpha=0.8)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{int(height)}', ha='center', fontsize=9)

plt.title("The number of patients with non-zero ICD codes")
plt.ylabel("Number of non-zero patients")
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

NameError: name 'icd_cols' is not defined

In [ ]:
# Optimize data types to significantly reduce memory usage
# 1. Number of visits column: Using int32 is sufficient to cover the maximum number.
visit_cols = ["total_visits", "telehealth_visits", "emergency_visits", "inpatient_visits", "outpatient_visits"]
diag_agg[visit_cols] = diag_agg[visit_cols].astype("int32")

# 2. ICD entry: Use int8 (count usually < 10)
icd_cols = [col for col in diag_agg.columns if col not in ["patient_id"] + visit_cols]
diag_agg[icd_cols] = diag_agg[icd_cols].astype("int8")

# Check the optimized memory usage
print(f"Optimized total memory usage: {diag_agg.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

## About <medications.csv>

In [ ]:
medications

In [ ]:
# 1. Core clinical features statistical aggregation
med_stats = medications.groupby("patient_id").agg(
    total_medications=("medication", "count"),
    avg_prescription_duration=("duration_days", "mean"),
    avg_adherence_pct=("adherence_pct", "mean"),
    low_adherence_count=("adherence_pct", lambda x: (x < 80).sum()),
    generic_med_ratio=("is_generic", "mean")
).reset_index()

# 2. One-hot for medication frequency (how often drugs are taken)
freq_dummies = pd.get_dummies(medications[["patient_id", "frequency"]], 
                              prefix="freq", columns=["frequency"])
freq_agg = freq_dummies.groupby("patient_id").max().reset_index()

# 3. One-hot for top 15 indications (why drugs are prescribed → diagnoses table)
top_indications = medications["indication"].value_counts().head(15).index
ind_subset = medications[medications["indication"].isin(top_indications)]
ind_dummies = pd.get_dummies(ind_subset[["patient_id", "indication"]], 
                             prefix="ind", columns=["indication"])
ind_agg = ind_dummies.groupby("patient_id").max().reset_index()

# 4. One-hot for top 15 medication names (which drugs patients use)
top_meds = medications["medication"].value_counts().head(15).index
med_subset = medications[medications["medication"].isin(top_meds)]
med_dummies = pd.get_dummies(med_subset[["patient_id", "medication"]], 
                             prefix="med", columns=["medication"])
med_agg_dummies = med_dummies.groupby("patient_id").max().reset_index()

# 5. Merge all medication features
med_agg = med_stats.merge(freq_agg, on="patient_id", how="left")
med_agg = med_agg.merge(ind_agg, on="patient_id", how="left")
med_agg = med_agg.merge(med_agg_dummies, on="patient_id", how="left")
med_agg = med_agg.fillna(0)

In [ ]:
med_agg

## About <lab_results.csv>

In [ ]:
lab_results

In [ ]:
# 1. Get latest lab results per patient and test
lab_sorted = lab_results.sort_values(["patient_id", "test_date"], ascending=[True, False])
lab_latest = lab_sorted.drop_duplicates(subset=["patient_id", "test_name"], keep="first")
# Avoiding the interference of historical data, it can better reflect the current health status.

# 2. Select top 20 most frequent tests
top_tests = lab_results["test_name"].value_counts().head(20).index
lab_wide = lab_latest.pivot(index="patient_id", columns="test_name", values="value").reset_index()
lab_wide = lab_wide[["patient_id"] + list(top_tests)].fillna(0)

# 3. Standardize lab values
scaler_lab = StandardScaler()
lab_wide[top_tests] = scaler_lab.fit_transform(lab_wide[top_tests])

# 4. Aggregate lab statistics
lab_stats = lab_results.groupby("patient_id").agg(
    abnormal_test_count=("is_abnormal", "sum"),
    abnormal_test_ratio=("is_abnormal", "mean")
).reset_index()

# 5. Merge stats and wide table
lab_agg = lab_stats.merge(lab_wide, on="patient_id", how="left").fillna(0)

In [ ]:
lab_agg

## About <outcomes.csv>

In [ ]:
outcomes

In [ ]:
# It's already patient-level data and no modification is needed.